In [18]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
)
df = pd.read_csv("Exam_Score_Prediction.csv")
print(df.head())

   student_id  age  gender   course  study_hours  class_attendance  \
0           1   17    male  diploma         2.78              92.9   
1           2   23   other      bca         3.37              64.8   
2           3   22    male     b.sc         7.88              76.8   
3           4   20   other  diploma         0.67              48.4   
4           5   20  female  diploma         0.89              71.6   

  internet_access  sleep_hours sleep_quality   study_method facility_rating  \
0             yes          7.4          poor       coaching             low   
1             yes          4.6       average  online videos          medium   
2             yes          8.5          poor       coaching            high   
3             yes          5.8       average  online videos             low   
4             yes          9.8          poor       coaching             low   

  exam_difficulty  exam_score  
0            hard        58.9  
1        moderate        54.8  
2       

In [19]:
# Adding pass_exam column
df["pass_exam"] = (df["exam_score"] >= 60).astype(int)
df = df.drop(columns=["student_id", "exam_score"])

In [20]:
# Кодирование
categorical_cols = [
    "gender",
    "course",
    "internet_access",
    "sleep_quality",
    "study_method",
    "exam_difficulty"
]

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

print(df)

       age  gender  course  study_hours  class_attendance  internet_access  \
0       17       1       6         2.78              92.9                1   
1       23       2       5         3.37              64.8                1   
2       22       1       1         7.88              76.8                1   
3       20       2       6         0.67              48.4                1   
4       20       0       6         0.89              71.6                1   
...    ...     ...     ...          ...               ...              ...   
19995   18       2       4         6.50              71.3                1   
19996   18       1       0         3.71              41.6                0   
19997   19       2       6         7.88              68.2                1   
19998   19       1       4         4.60              76.3                0   
19999   20       1       1         7.50              47.9                1   

       sleep_hours  sleep_quality  study_method facility_rating

In [16]:
# декартовый система координат
X = df.drop("pass_exam", axis=1)
y = df["pass_exam"]

In [21]:
# Training
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# колонки по типам
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns
num_cols = X.select_dtypes(include=[np.number]).columns

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ],
    remainder="drop"
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=2000))
])

# train
model.fit(X_train, y_train)

# predict
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.833
              precision    recall  f1-score   support

           0       0.81      0.82      0.81      1791
           1       0.85      0.85      0.85      2209

    accuracy                           0.83      4000
   macro avg       0.83      0.83      0.83      4000
weighted avg       0.83      0.83      0.83      4000



/tmp/ipykernel_19567/3095607130.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns
